In [ ]:
import pandas as pd

factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

# load just the datasets q01 needs:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
# Directly load the region table, removing the extra function call
region = pd.read_csv(
    f"{DATA_ROOT}/region.tbl",
    sep="|",
    storage_options=STORAGE_OPTS
)

In [ ]:
### cell 0 ###

nation_filtered = nation.loc[:, ["N_NATIONKEY", "N_NAME", "N_REGIONKEY"]]

In [ ]:
### cell 1 ###

region_filtered = region[(region["R_NAME"] == "EUROPE")]
region_filtered = region_filtered.loc[:, ["R_REGIONKEY"]]

In [ ]:
### cell 2 ###

r_n_merged = nation_filtered.merge(
    region_filtered, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
)
r_n_merged = r_n_merged.loc[:, ["N_NATIONKEY", "N_NAME"]]

In [ ]:
### cell 3 ###

supplier_filtered = supplier.loc[
    :,
    [
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_NATIONKEY",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]

In [ ]:
### cell 4 ###

s_r_n_merged = r_n_merged.merge(
    supplier_filtered, left_on="N_NATIONKEY", right_on="S_NATIONKEY", how="inner"
)

In [ ]:
### cell 5 ###

s_r_n_merged = s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]

In [ ]:
### cell 6 ###

partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]

In [ ]:
### cell 7 ###

ps_s_r_n_merged = s_r_n_merged.merge(
    partsupp_filtered, left_on="S_SUPPKEY", right_on="PS_SUPPKEY", how="inner"
)

In [ ]:
### cell 8 ###

ps_s_r_n_merged = ps_s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_PARTKEY",
        "PS_SUPPLYCOST",
    ],
]

In [ ]:
### cell 9 ###

part_filtered = part.loc[:, ["P_PARTKEY", "P_MFGR", "P_SIZE", "P_TYPE"]]

In [ ]:
### cell 10 ###

part_filtered = part_filtered[
    (part_filtered["P_SIZE"] == 15)
    & (part_filtered["P_TYPE"].str.endswith("BRASS"))
]

In [ ]:
### cell 11 ###

part_filtered = part_filtered.loc[:, ["P_PARTKEY", "P_MFGR"]]

In [ ]:
### cell 12 ###

merged_df = part_filtered.merge(
    ps_s_r_n_merged, left_on="P_PARTKEY", right_on="PS_PARTKEY", how="inner"
)

In [ ]:
### cell 13 ###

merged_df = merged_df.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_SUPPLYCOST",
        "P_PARTKEY",
        "P_MFGR",
    ],
]

In [ ]:
### cell 14 ###

min_values = merged_df.groupby("P_PARTKEY", as_index=False, sort=False)[
    "PS_SUPPLYCOST"
].min()
min_values.columns = ["P_PARTKEY_CPY", "MIN_SUPPLYCOST"]

In [ ]:
### cell 15 ###

merged_df = merged_df.merge(
    min_values,
    left_on=["P_PARTKEY", "PS_SUPPLYCOST"],
    right_on=["P_PARTKEY_CPY", "MIN_SUPPLYCOST"],
    how="inner",
)

In [ ]:
### cell 16 ###

total = merged_df.loc[
    :,
    [
        "S_ACCTBAL",
        "S_NAME",
        "N_NAME",
        "P_PARTKEY",
        "P_MFGR",
        "S_ADDRESS",
        "S_PHONE",
        "S_COMMENT",
    ],
]

In [ ]:
### cell 17 ###

total = total.sort_values(
    by=["S_ACCTBAL", "N_NAME", "S_NAME", "P_PARTKEY"],
    ascending=[False, True, True, True],
)